# SPARQL-RL (SRL) — User Guide

StarLayer provides support for SPARQL-RL.  SPARQL-RL (or SRL) is a general-purpose Datalog-style rules language published standalone at [SPARQL 1.2 RL](https://www.w3.org/TR/sparql12-rl/).  


## How to run this notebook

See [Getting Started](01-getting-started.ipynb) if you haven't installed StarLayer yet. This guide assumes StarLayer has been pip installed.

Run cells from top to bottom — later steps reuse variables from earlier ones.

## Step 1 — a base graph

A small software-dependency graph — the same running example [the SRL spec's own §6.6 worked example](https://www.w3.org/TR/sparql12-rl/#worked-example) uses: a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability; a `:logger` component has no vulnerabilities at all. The rules below will propagate the vulnerability along the dependency chain, flag critically-exposed components, and mark whatever's left over as safe.

In [1]:
from starlayergraph import StarLayerGraph, Namespace

import starsparql
from starsparql.srl_grammar import parse_ruleset

EX = Namespace("http://example.org/")

base = StarLayerGraph()
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - stays safe in Step 4

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")
base.bind("ex", EX)

print(base.serialize(format="turtle"))

@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:frontend a ex:Component ;
    ex:dependsOn ex:app .

ex:logger a ex:Component .

ex:app a ex:Component ;
    ex:dependsOn ex:db .

ex:db a ex:Component ;
    ex:hasVulnerability ex:vuln1 .

ex:vuln1 ex:severity 9.1 .




## Step 2 — write and parse an SRL rule

`RULE { head } WHERE { body }` — much like a SPARQL `CONSTRUCT`, but rules can depend on each other and even on their own output. R2 below is directly recursive: `:exposedTo` propagates through any number of `:dependsOn` hops. `parse_ruleset()` parses SRL text into a `starsparql.srl_ast.RuleSet` — a plain, inspectable Python dataclass tree, not an opaque object.

`ruleset_to_text()` renders it straight back to SRL text — no RDF involved at all (that's a separate representation, covered in Step 6). Useful for checking a rule set was parsed as intended, or for programmatically building/editing a rule set as data (the dataclasses) and getting readable text back out.

In [2]:
from starsparql.srl_to_text import ruleset_to_text

rules_text = """
PREFIX ex: <http://example.org/>

# R1: a component is exposed to any vulnerability it has directly.
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }

# R2: exposure propagates along dependsOn (recursive - depends on itself).
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }
"""

ruleset = parse_ruleset(rules_text)
print(f"{len(ruleset.rules)} rules parsed")
print(ruleset_to_text(ruleset, namespace_manager=base.namespace_manager))

2 rules parsed
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:hasVulnerability ?v . }
RULE { ?x ex:exposedTo ?v . } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }


## Step 3 — run inference

`srl_infer(base_graph, ruleset)` evaluates the rule set and returns a *new* graph containing only the newly-inferred triples — it never includes anything already in `base_graph` (see the [SRL spec's own §6](https://www.w3.org/TR/sparql12-rl/#rule-set-evaluation)). Named `srl_infer`, not the spec's own bare `Infer`, to avoid colliding with `StarLayerGraph.infer(profile=...)` (RDF/RDFS/OWL-RL/OWL-DL entailment — see the [Inferencing guide](02b-graphs-inferencing.ipynb)) if both are imported together. R2 is genuinely recursive, so this runs to a fixpoint internally — repeated passes until no new triples appear.

In [3]:
from starsparql.srl_eval import srl_infer

inferred = srl_infer(base, ruleset)
inferred.bind("ex", EX)
print(inferred.serialize(format="turtle"))

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 .

ex:db ex:exposedTo ex:vuln1 .

ex:frontend ex:exposedTo ex:vuln1 .




## Step 4 — `FILTER`, negation, and why rule order doesn't matter

Two more rules: R3 flags a component as critically exposed if any of its vulnerabilities has `severity >= 9.0` (`FILTER`); R4 marks a component safe if it's *not* critically exposed (`NOT { }`, negation-as-failure) — the only component this ends up true for is `:logger`, which has no vulnerabilities at all. R4 genuinely depends on R3 having finished — SRL's own [stratification algorithm](https://www.w3.org/TR/sparql12-rl/#stratification) works this out automatically from the rules' own structure, so every component's exposure is fully propagated *before* R4 ever runs, even though R4 is written and listed last.

In [4]:
rules_text += """
# R3: severity >= 9.0 counts as a critical vulnerability.
RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

# R4: not critically exposed -> safe to deploy.
RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = parse_ruleset(rules_text)
inferred = srl_infer(base, ruleset)
inferred.bind("ex", EX)
print(inferred.serialize(format="turtle"))

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:logger ex:status ex:safeToDeploy .




## Step 5 — inspecting the dependency graph and stratification directly

`build_dependency_graph()`/`stratify()` are the same functions `srl_infer()` uses internally — calling them directly shows exactly why R4 waits: it has a *closed* dependency on R3 (through the `NOT { }`), so stratification puts it in a later stratum than R1/R2/R3.

In [5]:
from starsparql.srl_eval import build_dependency_graph, stratify

for edge in build_dependency_graph(ruleset):
    print(ruleset.rules.index(edge.source), "->", ruleset.rules.index(edge.target), edge.label)

print()
for i, (once, general) in enumerate(stratify(ruleset)):
    once_idx = [ruleset.rules.index(r) for r in once]
    general_idx = [ruleset.rules.index(r) for r in general]
    print(f"stratum {i}: once={once_idx} general={general_idx}")

1 -> 0 open
1 -> 1 open
2 -> 0 open
2 -> 1 open
3 -> 2 closed

stratum 0: once=[] general=[0, 1, 2]
stratum 1: once=[] general=[3]


## Step 6 — encode the rule set as RDF

A second, separate representation from Step 2's text read-out: same "AST-as-RDF" pattern as [SPARQL queries themselves](05c-sparql-query-as-rdf.ipynb) — `ruleset_to_rdf()` encodes the whole `RuleSet` — every rule, triple pattern, filter expression, everything — into an ordinary RDF graph under the `srl:` namespace, so a rule set can be stored, queried, and edited as data just like anything else. `rdf_to_ruleset()` decodes it straight back, round-tripping exactly.

In [6]:
from starsparql.srl_from_rdf import rdf_to_ruleset
from starsparql.srl_to_rdf import ruleset_to_rdf

srl_graph, root = ruleset_to_rdf(ruleset)
print("encoded triples:", len(srl_graph))

round_tripped = rdf_to_ruleset(srl_graph, root)
print("round-trips exactly:", round_tripped == ruleset)

encoded triples: 119
round-trips exactly: True


## Step 7 — validate the encoding with SHACL

`starlayer.sparql.srl_shapes.validate_ruleset()` checks that an `srl:` graph is structurally well-formed — the same role `salg:` shapes play for encoded SPARQL queries. Useful for checking an LLM- or hand-authored `srl:` graph before decoding it.

In [ ]:
from starlayer.sparql.srl_shapes import validate_ruleset

conforms, _report_graph, report_text = validate_ruleset(srl_graph)
print("conforms:", conforms)

## Step 8 — catching a malformed rule before evaluating it

`check_ruleset()` implements SRL's own [§4.2 Well-formedness Conditions](https://www.w3.org/TR/sparql12-rl/#well-formedness-conditions) — every variable used in a rule's head must actually be bound by its own body. This is a real, easy mistake to make by hand (a typo'd variable name) — `check_ruleset()` catches it before `srl_infer()` ever runs on it.

In [8]:
from starsparql.srl_semantic_checks import check_ruleset

broken = parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:flaggedBy ?reviewer } WHERE { ?x ex:hasVulnerability ?v . }
""")

for issue in check_ruleset(broken):
    print(issue.kind, issue.variable, "in", issue.context)

unbound_head_variable reviewer in rule head


## Step 9 — `DATA` blocks and a genuine gotcha

A rule set can bundle its own ground facts in a `DATA { }` block, and a rule can scope its own matching to `WHERE DATA { }` instead of the ordinary base graph. **A real spec subtlety, confirmed directly against the raw spec text**: a `WHERE DATA { }` rule only ever matches the *original* base graph passed to `srl_infer()` — never the rule set's own `DATA { }` blocks, and never anything any rule (including itself) has already inferred. The `DATA { }` block's own facts still appear in the output (they're unconditionally included) — they're just invisible to `WHERE DATA` *matching* specifically, so the rule below derives nothing at all against an empty base graph.

In [9]:
data_rules = parse_ruleset("""
PREFIX ex: <http://example.org/>
DATA { ex:extra ex:seed ex:val . }
RULE { ?x ex:derivedFrom ?y } WHERE DATA { ?x ex:seed ?y . }
""")

empty_base = StarLayerGraph()
result = srl_infer(empty_base, data_rules)
for t in result:
    print(t)

(rdflib.term.URIRef('http://example.org/extra'), rdflib.term.URIRef('http://example.org/seed'), rdflib.term.URIRef('http://example.org/val'))


## Next steps

- [SHACL inference rules](04b-shacl-inference-rules.ipynb) — the RDF-native `sh:rule` alternative SRL is deliberately independent from.
- [SPARQL queries as RDF](05c-sparql-query-as-rdf.ipynb) — the same AST-as-RDF pattern, one layer over.
- `packages/sparql/CLAUDE.md`'s SRL section — full implementation notes, including every documented scope reduction (no `,`-chained object lists, no property paths beyond a bare IRI, `IMPORTS` is a hard rejection, and more).